# Epsilon Fund - Walk-Forward Validation
Uses `infrastructure/walk_forward/` to run rolling Optuna optimisation and evaluate OOS robustness. Ideal to use after finding strategy that seems to work using backtesting framework to ensure logic is valid.

---
### Iteration Guidelines

**Overfitting the iteration process:** Each time you inspect OOS results and adjust parameters, you leak OOS information into your design decisions. Cap yourself at **3–4 iterations** — first run with everything free, second with obvious fixes from CV + plateau analysis, third to tighten remaining params. 

If the strategy still shows heavy overfitting signals after three passes, **the problem is the strategy architecture, not the parameters**.

**WFE:** Walk-forward efficiency - examine IS/OOS ratio (simplest).

**Pertubation degradation:** Examine pertubation table to see if degradation reduces across runs.

| Signal | Meaning | Action |
|--------|---------|--------|
| IS Sharpe drops, OOS Sharpe holds or rises, WFE improves | Removing noise-fitting degrees of freedom | Continue iterating |
| Perturbation degradation shrinks across iterations | Parameters becoming more robust | Continue iterating |
| N/A plateau params decreasing across iterations | Strategy becoming more tolerant of parameter movement | Continue iterating |
| WFE improvement flattens (e.g. 0.55 → 0.65 → 0.67) | Diminishing returns — further fixes won't help much | Stop iterating |
| IS and OOS both decline but WFE rises (IS falls faster) | Constraining away real signal, not just noise | Stop iterating |
| OOS Sharpe keeps declining despite "better" param setup | Overfitting the iteration process itself | Stop — problem is strategy architecture, not parameters |
| WFE decreases after fixing a parameter | Locked in a param that was legitimately adapting across folds | Unfix that parameter and re-run |

---

In [6]:
import sys
import os
import pandas as pd
import numpy as np

# ── repo root — works on both Mac and Windows ────────────────────────────────
ROOT = os.path.expanduser('~/Desktop/epsilon/github/Epsilon-Quant-Research')
# ROOT = r'C:\Users\user\Documents\Epsilon Fund\Epsilon-Quant-Research'  # ← Windows path
# ─────────────────────────────────────────────────────────────────────────────

sys.path.append(os.path.join(ROOT, 'infrastructure', 'data'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'walkforward'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'backtester'))



from binance_client import get_binance_client, get_data
from wf_engine import walk_forward, plateau_analysis, plateau_summary, perturbation_test, cost_stress_test
from wf_visualizer import plot_walk_forward_results, plot_plateau_analysis
from engine import backtest

---
## Data

**Pairs** — any Binance pair in `BASEQUOTE` format (e.g. `BTCUSDT`, `ETHUSDT`, `SOLUSDT`, `BNBUSDT`).  
Verify availability at [binance.com/en/trade](https://www.binance.com/en/trade).

**Intervals** — `'1m'` `'5m'` `'15m'` `'1h'` `'4h'` `'1d'` `'1w'`

**Lookback** — days of history: must be >= (train_bars + test_bars) * n_folds desired

In [7]:
SYMBOL   = 'NEARUSDT'
INTERVAL = '1h'
LOOKBACK = 2151   # days → ~48 000 1H bars  (TRAIN=17520 + TEST=4380 = ~6 folds)


# ── Multiple pairs (optional) ──────────────────────────────────────────────────
# SYMBOLS = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']
# data_dict = get_multiple_data(client, SYMBOLS, INTERVAL, LOOKBACK)
# Access via: data_dict['BTCUSDT_1d'], data_dict['ETHUSDT_1d'] ...
# ──────────────────────────────────────────────────────────────────────────────

client   = get_binance_client()
df = get_data(client, SYMBOL, INTERVAL, LOOKBACK)
print(f'Data: {df.index[0].date()} → {df.index[-1].date()}  ({len(df)} bars)')
df.tail(3)

Data: 2020-10-14 → 2026-04-22  (48374 bars)


,Open,High,Low,Close,Volume
Time,,,,,
2026-04-22 12:00:00,1.425,1.428,1.413,1.415,417407.9
2026-04-22 13:00:00,1.415,1.424,1.410,1.419,620957.5
2026-04-22 14:00:00,1.420,1.426,1.416,1.423,165443.5


---
## Parameter Configuration

Define which parameters to optimise and anchor - **recommended to do after strategy writeup**

`FIXED_PARAMS`: choose parameters with CV < 0.15 from prior stability run, cross referencing with pertubation verdict table to reduce search space, improve OOS credibility.



In [18]:
# ── search ranges ─────────────────────────────────────────────────────────────
# Format: 'param_name': ('int' | 'float', lo, hi)
PARAM_DEFS = {
    # ── Bollinger Bands (bb_std removed — cancels in expansion ratio) ──────────
    'bb_period':         ('int',   10, 40),
    'bb_exp_window':     ('int',   2,  20),

    # ── ATR ────────────────────────────────────────────────────────────────────
    'atr_period':        ('int',   5,  20),

    # ── Breakout filter: percentile replaces fixed ATR multiple ───────────────
    # big candle = range > rolling quantile(breakout_pct) over breakout_lookback
    'breakout_pct':      ('float', 0.50, 0.85),
    'breakout_lookback': ('int',   20, 100),  # 4H bars for quantile window

    # ── MA periods decoupled: 4H trend filter vs 1H pullback anchor ───────────
    'h4_ma_period':      ('int',   10, 50),
    'slope_epsilon':     ('float', 0.0, 0.003), # allow slight negative slope (normalised by price)
    'h1_ma_period':      ('int',   5,  17),

    # ── Pullback filter split: entry zone and overshoot expiry independent ─────
    'entry_zone_bps':    ('int',   5,  100),   # P1 zone width around 1H SMA
    'overshoot_bps':     ('int',   5,  150),   # E3 max penetration past SMA

    # ── Other 1H filters ──────────────────────────────────────────────────────
    'max_1h_bars':       ('int',   12, 48),
    'pullback_atr_mult': ('float', 1.0, 3.0),
    'trail_atr_mult':    ('float', 0.5, 4.0),   # ATR multiple for trailing stop

    'adx_period':        ('int',   7, 21),     # fixed at 14 — in PARAM_DEFS for wf_engine visibility
    # ── ADX regime classifier ─────────────────────────────────────────────────
    # adx_strong: above this = strong trend → in_strong_bull active, shorts need -DI > +DI
    # adx_min:    below this = ranging noise → block shorts (no directional edge)
    'adx_strong':        ('float', 20.0, 60.0), # ADX threshold for strong trend

    # ── Regime-conditional TP ─────────────────────────────────────────────────
    # Strong bull (price > trend MA, ADX strong, +DI > -DI): pure trailing stop
    # Otherwise: tp_mult:1 TP cap — protect capital in ranging/bear regimes

    # ── Macro regime gate: long-term MA period (fixed at 200 via FIXED_PARAMS) ──
    'trend_ma_period':   ('int',   150, 300),   # 1H bars; fixed at 200 — here for wf_engine visibility

    # ── Position sizing ────────────────────────────────────────────────────────
    'risk_per_trade':    ('float', 0.01, 0.05),
    'max_leverage':      ('float', 1.0,  3.0),
}

# ── fixed params ──────────────────────────────────────────────────────────────
FIXED_PARAMS = {
    'max_leverage':      2.5,
    'risk_per_trade':    0.03,     
    
    'overshoot_bps': 99,
    'pullback_atr_mult': 2.717,
    'trail_atr_mult': 2.7646,

    'breakout_pct': 0.7113,

    'atr_period': 5,
    'max_1h_bars':23,
    'trend_ma_period': 220,
 
    
}

### *Guide to parameter anchoring*

|  | Robust Plateau| Fragile Plateau |
|--|-------------------|-------------------|
| Low CV | Stable across folds and insensitive to exact value - keep free| Looks stable but is fitting the same noise patterns - fix at concensus|
| High CV | Parameter unimportant - fix at any reasonable value | Unstable across folds and sitting on a cliff - strong candidate to eliminate |

Copy-paste plateau analysis table above into fixed params section and decide manually on which to fix/keep free.c

---
## Strategy

**CRUCIAL** - Strategy logic needs to work well in backtesting notebook before running here, saves time not running walk-forward for a broken strategy.

**Available columns:** `Open` `High` `Low` `Close` `Volume`

**BTC_2_1 changes from BTC_3:** Momentum resumption entry filter added (replaces bare zone touch). Entry requires: (1) price within `entry_zone_bps` of 1H SMA, AND (2) `close > prev_close` for longs / `close < prev_close` for shorts — confirms the pullback has reversed before committing. Looser than P2 (pin bar wick test): fires on any directional close, not a specific candle geometry.

**Required output:** a `position` column — `1` long · `0` flat · `-1` short  
**Optional:** `position_size` column (0–1) to use fractional capital

> The engine applies a 1-bar execution lag automatically. Inside the strategy loop, use `prev` for signal decisions and `curr` for execution — no manual shifting needed.

**To implement your strategy:**
1. Write strategy logic — compute indicators, signals, and execution loop: use `param_name`for those to be searched
2. Update `indicator_cols` to list your longest-warmup indicators — the engine uses this to clean NaN rows after OOS trimming


In [9]:
def my_strategy(df_slice: pd.DataFrame, params: dict) -> pd.DataFrame:

    df = df_slice.copy()
    h1 = df.rename(columns=str.lower)

    # ── Resample 1H → 4H ──────────────────────────────────────────────────────
    h4 = h1.resample('4h').agg({
        'open': 'first', 'high': 'max', 'low': 'min',
        'close': 'last', 'volume': 'sum',
    }).dropna()

    # ── Indicator helpers ──────────────────────────────────────────────────────
    def _atr(d, period):
        hi, lo, cl = d['high'], d['low'], d['close']
        prev_cl = cl.shift(1)
        tr = pd.concat([(hi - lo), (hi - prev_cl).abs(), (lo - prev_cl).abs()], axis=1).max(axis=1)
        return tr.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()

    def _sma(d, period):
        return d['close'].rolling(period).mean()

    def _bb_width(d, period):
        mid = d['close'].rolling(period).mean()
        std = d['close'].rolling(period).std()
        return (std * 2) / mid.replace(0, np.nan)

    def _ma_slope(d, period):
        ma = d['close'].rolling(period).mean()
        return ma - ma.shift(1)

    def _candle_range(d):
        return d['high'] - d['low']

    def _adx(d, period):
        hi, lo, cl = d['high'], d['low'], d['close']
        prev_hi = hi.shift(1)
        prev_lo = lo.shift(1)
        prev_cl = cl.shift(1)
        tr = pd.concat([(hi - lo), (hi - prev_cl).abs(), (lo - prev_cl).abs()], axis=1).max(axis=1)
        plus_dm  = (hi - prev_hi).clip(lower=0).where((hi - prev_hi) > (prev_lo - lo), 0.0)
        minus_dm = (prev_lo - lo).clip(lower=0).where((prev_lo - lo) > (hi - prev_hi), 0.0)
        alpha = 1.0 / period
        atr_w    = tr.ewm(alpha=alpha,      min_periods=period, adjust=False).mean()
        plus_di  = 100 * plus_dm.ewm(alpha=alpha,  min_periods=period, adjust=False).mean() / atr_w.replace(0, np.nan)
        minus_di = 100 * minus_dm.ewm(alpha=alpha, min_periods=period, adjust=False).mean() / atr_w.replace(0, np.nan)
        dx  = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan)
        adx = dx.ewm(alpha=alpha, min_periods=period, adjust=False).mean()
        return adx, plus_di, minus_di


    # ── 4H indicators ─────────────────────────────────────────────────────────
    h4_atr   = _atr(h4, params['atr_period'])
    h4_range = _candle_range(h4)
    h4_bw    = _bb_width(h4, params['bb_period'])
    h4_slope = _ma_slope(h4, params['h4_ma_period'])

    h4_green = h4['close'] > h4['open']
    h4_red   = h4['close'] < h4['open']

    brk_threshold = h4_range.rolling(int(params['breakout_lookback'])).quantile(params['breakout_pct'])
    big           = h4_range > brk_threshold

    two_big_green = big & big.shift(1) & h4_green & h4_green.shift(1)
    two_big_red   = big & big.shift(1) & h4_red   & h4_red.shift(1)

    bb_exp        = h4_bw > h4_bw.rolling(params['bb_exp_window']).mean()

    h4_slope_norm = h4_slope / h4['close'].replace(0, np.nan)
    slope_eps     = params['slope_epsilon']
    h4_long  = two_big_green & bb_exp & (h4_slope_norm >= -slope_eps)
    h4_short = two_big_red   & bb_exp & (h4_slope_norm <=  slope_eps)

    h4_adx, h4_plus_di, h4_minus_di = _adx(h4, params['adx_period'])

    # ── 1H indicators ─────────────────────────────────────────────────────────
    h1_atr      = _atr(h1,  params['atr_period'])
    h1_range    = _candle_range(h1)
    h1_sma      = _sma(h1,  params['h1_ma_period'])
    trend_period = int(params['trend_ma_period'])
    h1_trend_ma  = h1['close'].rolling(trend_period).mean()
    
    h1_pos_size = (
        params['risk_per_trade'] / (h1_atr / h1['close'])
    ).clip(0.1, params['max_leverage'])

    h4_adx_1h      = h4_adx.shift(1).reindex(h1.index,      method='ffill').fillna(0.0)
    h4_plus_di_1h  = h4_plus_di.shift(1).reindex(h1.index,  method='ffill').fillna(0.0)
    h4_minus_di_1h = h4_minus_di.shift(1).reindex(h1.index, method='ffill').fillna(0.0)

    h4_long_1h  = h4_long.shift(1).reindex(h1.index,  method='ffill').fillna(False)
    h4_short_1h = h4_short.shift(1).reindex(h1.index, method='ffill').fillna(False)

    long_setup_fires  = h4_long_1h  & ~h4_long_1h.shift(1).fillna(False)
    short_setup_fires = h4_short_1h & ~h4_short_1h.shift(1).fillna(False)

    # ── Extract numpy arrays ───────────────────────────────────────────────────
    close_arr     = h1['close'].to_numpy()
    sma_arr       = h1_sma.to_numpy()
    range_arr     = h1_range.to_numpy()
    atr_arr       = h1_atr.to_numpy()
    pos_size_arr  = h1_pos_size.to_numpy()
    trend_ma_arr  = h1_trend_ma.to_numpy()
    adx_arr       = h4_adx_1h.to_numpy()
    plus_di_arr   = h4_plus_di_1h.to_numpy()
    minus_di_arr  = h4_minus_di_1h.to_numpy()
    long_fire     = long_setup_fires.to_numpy()
    short_fire    = short_setup_fires.to_numpy()

    max_1h_bars       = params['max_1h_bars']
    pullback_atr_mult = params['pullback_atr_mult']
    entry_zone_bps    = params['entry_zone_bps']
    overshoot_bps     = params['overshoot_bps']

    # ── State machine — bar by bar ─────────────────────────────────────────────
    n             = len(h1)
    position      = np.zeros(n, dtype=int)
    position_size = np.ones(n)
    stop_loss     = np.zeros(n)

    setup_active    = False
    setup_direction = 0
    bars_since      = 0

    in_trade        = False
    trade_direction = 0
    trade_stop      = 0.0
    trade_tp        = 0.0
    trade_size      = 1.0

    for i in range(1, n):

        # ── 1. Trade management (Recommendation 1 & 3 implemented here) ──────
        if in_trade:
            close      = close_arr[i]
            h1_at_i    = atr_arr[i]
            trail_mult = params['trail_atr_mult']

            if not np.isnan(h1_at_i):
                if trade_direction == 1:
                    trade_stop = max(trade_stop, close - trail_mult * h1_at_i)
                else:
                    trade_stop = min(trade_stop, close + trail_mult * h1_at_i)

            stop_hit = (
                (trade_direction ==  1 and trade_stop > 0 and close <= trade_stop) or
                (trade_direction == -1 and trade_stop > 0 and close >= trade_stop)
            )
            tp_hit = (
                (trade_direction ==  1 and trade_tp > 0 and close >= trade_tp) or
                (trade_direction == -1 and trade_tp > 0 and close <= trade_tp)
            )

            if stop_hit or tp_hit:
                in_trade = False
                # Recommendation 1: Removed re-entry logic. Setup window is not re-armed.
            else:
                position[i]      = trade_direction
                position_size[i] = trade_size
            continue

        # ── 2. Setup detection ────────────────────────────────────────────────
        if not setup_active:
            if long_fire[i]:
                setup_active, setup_direction, bars_since = True,  1, 0
            elif short_fire[i]:
                trend_ma_i = trend_ma_arr[i]
                adx_i      = adx_arr[i]
                plus_di_i  = plus_di_arr[i]
                minus_di_i = minus_di_arr[i]
                adx_strong = params['adx_strong']
                
                above_ma   = not np.isnan(trend_ma_i) and close_arr[i] > trend_ma_i
                bull_trend = adx_i > adx_strong and plus_di_i > minus_di_i
                if not above_ma and not bull_trend:
                    setup_active, setup_direction, bars_since = True, -1, 0

        if not setup_active:
            continue

        # ── 3. Expiry checks ──────────────────────────────────────────────────
        bars_since += 1
        close  = close_arr[i]
        s_ma   = sma_arr[i]
        h1_rng = range_arr[i]
        h1_at  = atr_arr[i]

        if np.isnan(s_ma) or np.isnan(h1_at) or s_ma == 0:
            continue

        if bars_since > max_1h_bars:
            setup_active = False
            continue

        if h1_rng > pullback_atr_mult * h1_at:
            setup_active = False
            continue

        if setup_direction == 1:
            if close < s_ma - (s_ma * overshoot_bps / 10000):
                setup_active = False
                continue
        else:
            if close > s_ma + (s_ma * overshoot_bps / 10000):
                setup_active = False
                continue

        # ── 4. Entry (Recommendation 2 & 3 implemented here) ─────────────────
        bps_from_sma = abs(close - s_ma) / s_ma * 10000
        in_zone = bps_from_sma <= entry_zone_bps

        prev_close = close_arr[i - 1]
        momentum_ok = (
            (setup_direction ==  1 and close > prev_close) or
            (setup_direction == -1 and close < prev_close)
        )

        if in_zone and momentum_ok:
            raw_size   = pos_size_arr[i]
            sz         = raw_size if not np.isnan(raw_size) else 1.0
            trail_mult = params['trail_atr_mult']

            # Recommendation 3: stop_atr_mult removed. Initial stop is the trail distance.
            initial_stop_dist = trail_mult * h1_at
            ts_val = (close - initial_stop_dist) if setup_direction == 1 else (close + initial_stop_dist)

            # Recommendation 2: Hard-coded TP logic
            in_strong_bull = (
                close > trend_ma_arr[i]
                and adx_arr[i] > params['adx_strong']
                and plus_di_arr[i] > minus_di_arr[i]
            )
            
            if in_strong_bull or np.isnan(trend_ma_arr[i]):
                tp_val = 0.0   # ride the trend via trailing stop only
            else:
                # Set fixed 6:1 TP ratio relative to the initial ATR-based risk
                tp_dist = 6 * initial_stop_dist
                tp_val = (close + tp_dist) if setup_direction == 1 else (close - tp_dist)

            position[i]      = setup_direction
            position_size[i] = sz
            stop_loss[i]     = ts_val

            in_trade        = True
            trade_direction = setup_direction
            trade_stop      = ts_val   
            trade_tp        = tp_val   
            trade_size      = sz
            setup_active    = False

    # ─────────────────────────────────────────────────────────────────────────
    indicator_cols      = ['SMA']
    df['SMA']           = h1_sma.to_numpy()
    df['position']      = position
    df['position_size'] = position_size
    df['stop_loss']     = stop_loss

    return df, indicator_cols

---
## Run Walk-Forward
Simulates how the strategy would have performed if re-optimised periodically
in live trading, and exposes whether good IS performance survives unseen data.

**Folds Setup**
| Parameter | Description | Guidance |
|---|---|---|
| `TRAIN_BARS` | Bars per training window | Aim for 2:1 to 3:1 ratio vs `TEST_BARS` |
| `TEST_BARS` | Bars per test window | `365` = ~1 year on daily data |
| `BURNIN_BARS` | Bars prepended to test for indicator warmup | Match your longest indicator period |
| `N_TRIALS` | Optuna trials per fold | 300–500 for daily; more = better but slower |
| `COST` | Round-trip cost per trade | `0.001` = 0.1% |

Use `N_TRIALS` as robustness dia: if OOS degrades sharply as you increase it from 100→200→300, direct signal your parameter space still has too many degrees of freedom relative to the information content of the training window (consider decreasing). 

**Folds Selection:** further robustness test, recommended to run with more folds to begin with as params robustness tests are more reliable, then validate that efficiency holds by reducing i.e test with 10 folds 100 periods, validate with 200 periods

**Score and Rejection** — use to calibrate what Optuna optimises IS: default `score_fn(m)` uses weighted basket of Sharpe, Calmar and Return, normalised using their "max" value; default `reject_fn(m)` discards runs failing certain criteria that limits credibility.

> Pay attention to the **degradation ratio** — OOS/IS Sharpe reveals overfitting.

In [19]:
# ── walk-forward windows ──────────────────────────────────────────────────────
TRAIN_BARS  = 25200   # ~2 years of 1H bars  (~7-8 trades/fold at observed frequency)
TEST_BARS   = 5770    # ~6 months of 1H bars  (3:1 train:test ratio)
BURNIN_BARS = 200     # indicator warmup (covers max bb_period×4 on resampled 4H)
N_TRIALS    = 400
COST        = 0.001

# ── SCORING FUNCTION ──────────────────────────────────────────────────────────
# Modify weights or swap components. Must return a float (higher = better).

def score_fn(m):
    SHARPE_MAX = 3
    CALMAR_MAX = 10.0
    RETURN_MAX = 100.0

    calmar = m['total_return'] / abs(m['max_drawdown']) if m['max_drawdown'] != 0 else 0

    s = np.clip(m['sharpe_ratio']  / SHARPE_MAX, 0, 1)
    c = np.clip(calmar             / CALMAR_MAX, 0, 1)
    r = np.clip(m['total_return']  / RETURN_MAX, 0, 1)

    return 0.50 * s + 0.30 * c + 0.20 * r

# ── REJECTION CRITERIA ────────────────────────────────────────────────────────
# Calibrated to strategy's observed trade frequency (~7-8 trades per 2yr IS window).

def reject_fn(m):
    if m is None:                     return True
    if m['num_trades']    < 15:        return True   # sparse strategy — lowered from 5
    if m['win_rate']      < 0.4:      return True
    if m['max_drawdown']  < -0.6:       return True
    if m['profit_factor'] < 0.5:      return True
    return False


results = walk_forward(
    df           = df,
    strategy_fn  = my_strategy,
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    train_bars   = TRAIN_BARS,
    test_bars    = TEST_BARS,
    burnin_bars  = BURNIN_BARS,
    n_trials     = N_TRIALS,
    cost         = COST,
    score_fn     = score_fn,    # ← your notebook definition
    reject_fn    = reject_fn,   # ← your notebook definition
    save_csv     = None,
)

UPDATED WALK_FORWARD FILE IS RUNNING
Walk-forward: 4 fold(s)  train=25200  test=5770  burnin=200  trials=400
  Fold 1: train 2020-10-14 → 2023-08-31  | test 2023-08-31 → 2024-04-27
  Fold 2: train 2021-06-12 → 2024-04-27  | test 2024-04-27 → 2024-12-23
  Fold 3: train 2022-02-07 → 2024-12-23  | test 2024-12-23 → 2025-08-21
  Fold 4: train 2022-10-06 → 2025-08-21  | test 2025-08-21 → 2026-04-18

Fixed (9): ['max_leverage', 'risk_per_trade', 'overshoot_bps', 'pullback_atr_mult', 'trail_atr_mult', 'breakout_pct', 'atr_period', 'max_1h_bars', 'trend_ma_period']
Free  (9): ['bb_period', 'bb_exp_window', 'breakout_lookback', 'h4_ma_period', 'slope_epsilon', 'h1_ma_period', 'entry_zone_bps', 'adx_period', 'adx_strong']

────────────────────────────────────────────────────────────
Fold 1/4  train: 2020-10-14 → 2023-08-31  test: 2023-08-31 → 2024-04-27


  0%|          | 0/400 [00:00<?, ?it/s]


  IS  → Sharpe: 1.67  Return: 1471.42%  DD: -51.31%  Calmar: 3.13  Trades: 80
  OOS → Sharpe: 2.17  Return: 153.90%  DD: -23.11%  Calmar: 13.48  Trades: 15

  Best params: {'max_leverage': 2.5, 'risk_per_trade': 0.03, 'overshoot_bps': 99, 'pullback_atr_mult': 2.717, 'trail_atr_mult': 2.7646, 'breakout_pct': 0.7113, 'atr_period': 5, 'max_1h_bars': 23, 'trend_ma_period': 220, 'bb_period': 18, 'bb_exp_window': 19, 'breakout_lookback': 20, 'h4_ma_period': 32, 'slope_epsilon': 0.0026927742413442824, 'h1_ma_period': 7, 'entry_zone_bps': 100, 'adx_period': 7, 'adx_strong': 49.41046388928681}

────────────────────────────────────────────────────────────
Fold 2/4  train: 2021-06-12 → 2024-04-27  test: 2024-04-27 → 2024-12-23


  0%|          | 0/400 [00:00<?, ?it/s]


  IS  → Sharpe: 1.70  Return: 827.98%  DD: -29.74%  Calmar: 3.93  Trades: 32
  OOS → Sharpe: 3.61  Return: 280.04%  DD: -27.76%  Calmar: 23.75  Trades: 12

  Best params: {'max_leverage': 2.5, 'risk_per_trade': 0.03, 'overshoot_bps': 99, 'pullback_atr_mult': 2.717, 'trail_atr_mult': 2.7646, 'breakout_pct': 0.7113, 'atr_period': 5, 'max_1h_bars': 23, 'trend_ma_period': 220, 'bb_period': 37, 'bb_exp_window': 3, 'breakout_lookback': 72, 'h4_ma_period': 33, 'slope_epsilon': 0.0011426850624906373, 'h1_ma_period': 11, 'entry_zone_bps': 20, 'adx_period': 18, 'adx_strong': 54.0426918745765}

────────────────────────────────────────────────────────────
Fold 3/4  train: 2022-02-07 → 2024-12-23  test: 2024-12-23 → 2025-08-21


  0%|          | 0/400 [00:00<?, ?it/s]


  IS  → Sharpe: 2.69  Return: 13662.42%  DD: -37.93%  Calmar: 11.97  Trades: 69
  OOS → Sharpe: -0.26  Return: -18.18%  DD: -36.30%  Calmar: -0.72  Trades: 18

  Best params: {'max_leverage': 2.5, 'risk_per_trade': 0.03, 'overshoot_bps': 99, 'pullback_atr_mult': 2.717, 'trail_atr_mult': 2.7646, 'breakout_pct': 0.7113, 'atr_period': 5, 'max_1h_bars': 23, 'trend_ma_period': 220, 'bb_period': 18, 'bb_exp_window': 5, 'breakout_lookback': 25, 'h4_ma_period': 35, 'slope_epsilon': 0.0007206046519326247, 'h1_ma_period': 7, 'entry_zone_bps': 44, 'adx_period': 17, 'adx_strong': 46.09017600142666}

────────────────────────────────────────────────────────────
Fold 4/4  train: 2022-10-06 → 2025-08-21  test: 2025-08-21 → 2026-04-18


  0%|          | 0/400 [00:00<?, ?it/s]


  IS  → Sharpe: 2.31  Return: 6785.56%  DD: -37.48%  Calmar: 8.95  Trades: 77
  OOS → Sharpe: 0.79  Return: 20.72%  DD: -24.23%  Calmar: 1.37  Trades: 17

  Best params: {'max_leverage': 2.5, 'risk_per_trade': 0.03, 'overshoot_bps': 99, 'pullback_atr_mult': 2.717, 'trail_atr_mult': 2.7646, 'breakout_pct': 0.7113, 'atr_period': 5, 'max_1h_bars': 23, 'trend_ma_period': 220, 'bb_period': 18, 'bb_exp_window': 15, 'breakout_lookback': 31, 'h4_ma_period': 37, 'slope_epsilon': 0.0008282377543581022, 'h1_ma_period': 7, 'entry_zone_bps': 65, 'adx_period': 12, 'adx_strong': 48.33775563235247}

════════════════════════════════════════════════════════════
WALK-FORWARD SUMMARY
════════════════════════════════════════════════════════════

Out-of-sample across 4 fold(s):
  Avg Sharpe:       1.58
  Avg Return:       109.1%
  Avg Max Drawdown: -27.9%
  Avg Calmar:       9.47
  Avg Trades/fold:  16
  Folds profitable: 3/4
  Sharpe OOS/IS:    0.75  (good)

───────────────────────────────────────────────

---
## Granular Results and Parameter Stability

Per-fold IS vs OOS performance. Each row is one fold — compare `train_*` vs `test_*` columns to assess overfitting.

| Column | Description |
|---|---|
| `*_sharpe` `*_return` `*_drawdown` `*_calmar` | Core performance metrics |
| `*_trades` `*_winrate` `*_profit_factor` | Trade statistics |
| `optuna_score` | Best score achieved on training window |
| `param_*` | Best parameter values per fold e.g. `param_ema_span` |

**Concensus Parameters** - use to anchor: the engine determines stability using the coefficient of variation (CV) — the standard deviation of a parameter's best values across all folds divided by their median.

>CV < 0.15: indicates the strategy  relies on value rather than it being noise-fitted to a specific period — making it safe to fix for future runs. A high CV means the parameter is period-sensitive and should stay free.

In [22]:
# ── fold summary table ────────────────────────────────────────────────────────
display_cols = [
    'train_start', 'train_end',
    'test_start', 'test_end',
    'train_sharpe', 'test_sharpe',
    'train_return', 'test_return',
    'train_drawdown', 'test_drawdown',
    'test_trades',  'test_winrate', 'optuna_score',
]
display(results['results_df'][display_cols].round(2))

# ── parameter stability ───────────────────────────────────────────────────────
stability = results['stability_df'].copy()
stability['stable'] = stability['stable'].map({True: '✓', False: ''})
stability['fixed']  = stability['fixed'].map({True: '✓', False: ''})
stability = stability[['param', 'median', 'std', 'cv', 'stable', 'fixed']].round(2)
display(stability.sort_values('cv'))

# ── consensus params ──────────────────────────────────────────────────────────
stable = results['stability_df'][results['stability_df']['cv'] < 0.15]

print('Stable parameters (CV < 0.15) — copy into FIXED_PARAMS:')
for _, row in stable.iterrows():
    v = results['consensus_params'][row['param']]
    v_fmt = int(round(v)) if isinstance(v, float) and v == int(v) else round(v, 4) if isinstance(v, float) else v
    print(f"    '{row['param']}': {v_fmt},")
    
print('\nConsensus parameters (median across folds):')
for k, v in results['consensus_params'].items():
    print(f'  {k:<30} = {round(v, 2) if isinstance(v, float) else v}')

,train_start,train_end,test_start,test_end,train_sharpe,test_sharpe,train_return,test_return,train_drawdown,test_drawdown,test_trades,test_winrate,optuna_score
0,2020-10-14,2023-08-31,2023-08-31,2024-04-27,1.67,2.17,14.71,1.54,-0.51,-0.23,15,0.67,0.61
1,2021-06-12,2024-04-27,2024-04-27,2024-12-23,1.70,3.61,8.28,2.80,-0.30,-0.28,12,0.67,0.60
2,2022-02-07,2024-12-23,2024-12-23,2025-08-21,2.69,-0.26,136.62,-0.18,-0.38,-0.36,18,0.39,0.95
3,2022-10-06,2025-08-21,2025-08-21,2026-04-18,2.31,0.79,67.86,0.21,-0.37,-0.24,17,0.41,0.82


,param,median,std,cv,stable,fixed
17,max_leverage,2.50,0.00,0.00,✓,✓
15,trend_ma_period,220.00,0.00,0.00,✓,✓
2,atr_period,5.00,0.00,0.00,✓,✓
3,breakout_pct,0.71,0.00,0.00,✓,✓
16,risk_per_trade,0.03,0.00,0.00,✓,✓
9,overshoot_bps,99.00,0.00,0.00,✓,✓
10,max_1h_bars,23.00,0.00,0.00,✓,✓
11,pullback_atr_mult,2.72,0.00,0.00,✓,✓
12,trail_atr_mult,2.76,0.00,0.00,✓,✓
14,adx_strong,48.87,2.90,0.06,✓,


Stable parameters (CV < 0.15) — copy into FIXED_PARAMS:
    'atr_period': 5,
    'breakout_pct': 0.7113,
    'h4_ma_period': 34,
    'overshoot_bps': 99,
    'max_1h_bars': 23,
    'pullback_atr_mult': 2.717,
    'trail_atr_mult': 2.7646,
    'adx_strong': 48.8741,
    'trend_ma_period': 220,
    'risk_per_trade': 0.03,
    'max_leverage': 2.5,

Consensus parameters (median across folds):
  bb_period                      = 18
  bb_exp_window                  = 10
  atr_period                     = 5
  breakout_pct                   = 0.71
  breakout_lookback              = 28
  h4_ma_period                   = 34
  slope_epsilon                  = 0.0
  h1_ma_period                   = 7
  entry_zone_bps                 = 54
  overshoot_bps                  = 99
  max_1h_bars                    = 23
  pullback_atr_mult              = 2.72
  trail_atr_mult                 = 2.76
  adx_period                     = 14
  adx_strong                     = 48.87
  trend_ma_period             

---
## Parameter Robustness Checks

### Plateau Analysis
Sweep each free parameter across its range while holding others at consensus (median) value then evaluates the `score` at each point by backtesting over entire lookback.

The stability table (CV across folds) tells you *"does the optimizer consistently pick the same value?"*  

Plateau analysis tells you *"if that value were slightly wrong, would performance collapse?"*  

**Plateau %** - what fraction of each parameter's range stays within `threshold`% (default 20) of peak score: >60% = `robust plateau`, 30–60% = `moderate`, <30% = `fragile` (consider anchoring). `N/A` means every sweep point failed rejection filters — the strategy is completely intolerant of movement on that dimension.

>Run time: `n_free_params` × `n_steps`

### Perturbation test
Jitters all free parameters by ±5/10/20% of their range (50 random samples per offset range). Measures how much the score degrades vs the base

Test whether optimum is a broad hill in `#free params`-D space or a narrow spike

**>15%:** fragile optimum, consider reducing free parameters

In [23]:
# ── 1-D sensitivity sweeps around consensus params ─────────────────────────
sweep_results = plateau_analysis(
    df           = df,
    strategy_fn  = my_strategy,
    base_params  = results['consensus_params'],
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    cost         = COST,
    n_steps      = 20, #Adjust for number of steps around concensus per parameter
)

# ── text verdicts ──────────────────────────────────────────────────────────
verdict_df = plateau_summary(
    sweep_results,
    base_params = results['consensus_params'],
    stability_df = results['stability_df'],  
    threshold   = 0.20, #Adjust for % around peak score
)

# ── neighbouahood perturbation ────────────────────────────────────────────
# Randomly jitters ALL free params simultaneously.
# If mean score degrades >15% at ±10% offset, the optimum is fragile.

perturb_df = perturbation_test(
    df           = df,
    strategy_fn  = my_strategy,
    base_params  = results['consensus_params'],
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    cost         = COST,
    pct_offsets  = (0.05, 0.10, 0.20),   # ±5%, ±10%, ±20% of range
    n_samples    = 50,                     # random perturbations per offset level
)


═══════════════════════════════════════════════════════════════════════════════════════════
PLATEAU ANALYSIS — PARAMETER ROBUSTNESS
═══════════════════════════════════════════════════════════════════════════════════════════
Parameter                 Consensus Peak Score  Plateau %  Fold CV Verdict                 
──────────────────────── ────────── ────────── ────────── ──────── ────────────────────────
bb_exp_window                    10      0.801      100.0%    0.669 Robust                  
slope_epsilon                0.0010      0.797      100.0%    0.805 Robust                  
h4_ma_period                     34      0.794      100.0%    0.056 Robust                  
adx_period                       14      0.786      100.0%    0.303 Robust                  
adx_strong                  48.8741      0.786      100.0%    0.059 Robust                  
entry_zone_bps                   54      0.809       75.0%    0.539 Robust                  
breakout_lookback                

### 1-D sweep charts:
| Element | Meaning | Good | Bad |
|---------|---------|------|-----|
| **Blue curve** | Composite score at each value of the parameter, with all others held at consensus | Flat-topped curve — performance is insensitive to the exact value | Narrow spike — optimizer latched onto one specific value, everything nearby is worse |
| **Red dashed line** | Where the consensus value sits | On the flat top of the curve | On a steep slope or at the edge of a cliff |
| **Green dashed line** | Cutoff at 80% of peak score — the boundary between plateau and non-plateau | Blue curve stays above this line across most of the range | Blue curve dips below it quickly either side of the peak |
| **Green shading** | Plateau region — all values where the score stays within 20% of the peak | Wide green band spanning most of the range (robust) | Thin sliver or no shading at all (fragile/overfit) |


In [24]:
# ── visual sweep curves ───────────────────────────────────────────────────
plot_plateau_analysis(
    sweep_results    = sweep_results,
    consensus_params = results['consensus_params'],
    param_defs       = PARAM_DEFS,
    fixed_params     = FIXED_PARAMS,
    threshold        = 0.20,
    show             = False,
    save_html        = None,
)

---
## Results Charts and Cost Stress Test

| Parameter | Description | Default |
|---|---|---|
| `show_fold_perf` | IS vs OOS bars for return, Sharpe, drawdown per fold | `False` |
| `show_param_evol` | Parameter evolution across folds with ±1 std bands | `False` |
| `show_oos_equity` | Combined OOS equity curve + drawdown with fold boundaries | `True` |
| `show_trades` | Overlay entry/exit markers on OOS equity chart | `False` |
| `benchmark_data` | DataFrame with `Close` column for buy & hold comparison | `None` |
| `save_html_dir` | Directory path to save charts as HTML files, or `None` | `None` |

**Cost Stress Test:** re-run the combined OOS backtest at 1×, 1.5×, 2×, 3× the base cost. Fragile strategies collapse; robust ones degrade gradually.

In [20]:
plot_walk_forward_results(
    results          = results,
    param_defs       = PARAM_DEFS,
    fixed_params     = FIXED_PARAMS,
    benchmark_data   = df,
    show             = True,
    save_html_dir    = None,
    show_fold_perf   = False,   # IS vs OOS bars by fold
    show_param_evol  = False,   # parameter evolution across folds
    show_oos_equity  = True,   # combined OOS equity curve
    show_trades      = False,  # trade markers on OOS equity chart
)

# ── transaction cost stress test ──────────────────────────────────────────

if results['oos_combined_df'] is not None:
    cost_df = cost_stress_test(
        oos_combined_df  = results['oos_combined_df'],
        cost_multipliers = (1.0, 1.5, 2.0, 3.0),
        base_cost        = COST,
    )
else:
    print('No combined OOS dataframe — skip cost stress test')


═══════════════════════════════════════════════════════════════════════════
TRANSACTION COST STRESS TEST
═══════════════════════════════════════════════════════════════════════════
    Cost   Mult   Sharpe     Return      MaxDD   Calmar       PF
──────── ────── ──────── ────────── ────────── ──────── ────────
  0.0010   1.0x     1.65    853.09%    -47.44%     2.85     2.64
  0.0015   1.5x     1.62    794.98%    -48.08%     2.70     2.64
  0.0020   2.0x     1.58    740.39%    -48.71%     2.55     2.64
  0.0030   3.0x     1.50    640.93%    -49.95%     2.28     2.64


In [21]:
results['oos_combined_df'].to_pickle('near_oos.pkl')